In [1]:
from ready_for_ML import FootballPreprocessor
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.metrics import classification_report


In [1]:
from ready_for_ML import FootballPreprocessor
from walk_forward import WalkForwardValidator

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


processor = FootballPreprocessor([
    "dataset/21-22.csv",
    "dataset/22-23.csv",
    "dataset/23-24.csv",
    "dataset/24-25.csv",
    "dataset/25-26.csv"
])


df = processor.load_data()

X, y, meta = processor.prepare_features(df)


validator = WalkForwardValidator(
    meta["Season"].unique()
)


for train_idx, test_idx, train_seasons, test_season in validator.split(meta):


    print(
        "TRAIN:",
        train_seasons,
        "TEST:",
        test_season
    )


    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]

    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]


    # preprocessing AFTER split
    processor.fit(X_train)

    X_train = processor.transform(X_train)
    X_test = processor.transform(X_test)


    y_train = processor.encode_target(y_train)
    y_test = processor.transform_target(y_test)


    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    predictions = model.predict(X_test)


    print(classification_report(
        y_test,
        predictions
    ))

TRAIN: ['21-22'] TEST: 22-23
              precision    recall  f1-score   support

           0       0.42      0.51      0.46       105
           1       0.29      0.06      0.10        80
           2       0.59      0.73      0.65       175

    accuracy                           0.52       360
   macro avg       0.44      0.43      0.41       360
weighted avg       0.48      0.52      0.47       360

TRAIN: ['21-22', '22-23'] TEST: 23-24
              precision    recall  f1-score   support

           0       0.56      0.55      0.55       116
           1       0.44      0.05      0.09        78
           2       0.58      0.82      0.68       165

    accuracy                           0.57       359
   macro avg       0.53      0.48      0.44       359
weighted avg       0.54      0.57      0.51       359

TRAIN: ['21-22', '22-23', '23-24'] TEST: 24-25
              precision    recall  f1-score   support

           0       0.54      0.56      0.55       124
           1   

In [4]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(X_train,y_train)

print(dummy.score(X_test,y_test))

print(model.score(X_test,y_test))

0.4117647058823529
0.44982698961937717


In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")  # adjust path/filename

# --- Chronological season split ---
train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# --- Columns to drop from features (leakage / non-numeric identifiers) ---
drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["FTR"]

# --- Handle any missing values (early-season rolling stats will have NaNs) ---
print(f"NaNs in X_train: {X_train.isna().sum().sum()}")
print(f"NaNs in X_test: {X_test.isna().sum().sum()}")

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# --- Train ---
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

# --- Predict & evaluate ---
y_pred = rf.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(rf.classes_)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))

# --- Feature importance, since Elo is the whole point of this exercise ---
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print("\nTop 15 most important features:")
print(importances.sort_values(ascending=False).head(15))

Train shape: (1439, 148)
Test shape: (360, 148)
NaNs in X_train: 0
NaNs in X_test: 0

Classification Report:
              precision    recall  f1-score   support

           A       0.40      0.50      0.44       109
           D       0.27      0.04      0.07        99
           H       0.52      0.71      0.60       152

    accuracy                           0.46       360
   macro avg       0.39      0.42      0.37       360
weighted avg       0.41      0.46      0.41       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 54   5  50]
 [ 44   4  51]
 [ 38   6 108]]

Top 15 most important features:
Home_Elo                                     0.027191
Away_TablePosDiff                            0.016332
Home_TablePosDiff                            0.015207
Away_Elo                                     0.014335
Away_ShotDifference_Rolling5_all             0.012648
Home_Shots_Rolling10                         0.011414
Away_ShotOnTargetDiffe

In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_5roll.csv")  # adjust path/filename

# --- Chronological season split ---
train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# --- Columns to drop from features (leakage / non-numeric identifiers) ---
drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["FTR"]

# --- Handle any missing values (early-season rolling stats will have NaNs) ---
print(f"NaNs in X_train: {X_train.isna().sum().sum()}")
print(f"NaNs in X_test: {X_test.isna().sum().sum()}")

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# --- Train ---
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

# --- Predict & evaluate ---
y_pred = rf.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(rf.classes_)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))

# --- Feature importance, since Elo is the whole point of this exercise ---
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print("\nTop 15 most important features:")
print(importances.sort_values(ascending=False).head(15))

Train shape: (1439, 54)
Test shape: (360, 54)
NaNs in X_train: 104
NaNs in X_test: 4

Classification Report:
              precision    recall  f1-score   support

           A       0.38      0.48      0.42       109
           D       0.32      0.06      0.10        99
           H       0.52      0.70      0.60       152

    accuracy                           0.46       360
   macro avg       0.40      0.41      0.37       360
weighted avg       0.42      0.46      0.41       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 52   8  49]
 [ 44   6  49]
 [ 41   5 106]]

Top 15 most important features:
Elo_Difference                          0.065215
Home_Elo                                0.053583
Home_TablePosDiff                       0.035551
ShotDifference_Rolling5                 0.032248
Away_Elo                                0.031796
TablePosDiff_Rolling5                   0.031719
Away_TablePosDiff                       0.028035
Hom

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta):
        unique_seasons = sorted(meta["Season"].unique())

        for i in range(1, len(unique_seasons)):
            train_seasons = unique_seasons[:i]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, test_idx, train_seasons, test_season)


# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_5roll.csv")
all_df = all_df.reset_index(drop=True)  # important: validator yields positional-style index, keep it clean

print(f"Full dataset shape: {all_df.shape}")
print(f"Seasons found: {sorted(all_df['Season'].unique())}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_results = []
all_importances = []
class_labels = sorted(all_df["FTR"].unique())  # fixed label order across folds for aggregation

for fold_num, (train_idx, test_idx, train_seasons, test_season) in enumerate(validator.split(all_df), start=1):

    train_df = all_df.loc[train_idx]
    test_df = all_df.loc[test_idx]

    print(f"\n{'='*60}")
    print(f"Fold {fold_num}: train on {train_seasons} -> test on {test_season}")
    print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

    X_train = train_df.drop(columns=drop_cols)
    y_train = train_df["FTR"]

    X_test = test_df.drop(columns=drop_cols)
    y_test = test_df["FTR"]

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f} | Macro F1: {f1_macro:.4f}")
    print(classification_report(y_test, y_pred, labels=class_labels))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(class_labels)
    print(confusion_matrix(y_test, y_pred, labels=class_labels))

    fold_results.append({
        "fold": fold_num,
        "test_season": test_season,
        "train_seasons": train_seasons,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "n_train": len(train_df),
        "n_test": len(test_df),
    })

    all_importances.append(pd.Series(rf.feature_importances_, index=X_train.columns))

# --- Aggregate across folds ---
results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print("WALK-FORWARD SUMMARY")
print(f"{'='*60}")
print(results_df[["fold", "test_season", "n_train", "n_test", "accuracy", "f1_macro"]])

print(f"\nMean accuracy across folds: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
print(f"Mean macro F1 across folds: {results_df['f1_macro'].mean():.4f} (+/- {results_df['f1_macro'].std():.4f})")

# --- Aggregate feature importance across folds ---
importance_df = pd.concat(all_importances, axis=1)
importance_df.columns = [f"fold_{r['fold']}_{r['test_season']}" for r in fold_results]
mean_importance = importance_df.mean(axis=1).sort_values(ascending=False)

print("\nTop 15 features by mean importance across all folds:")
print(mean_importance.head(15))

Full dataset shape: (1799, 54)
Seasons found: ['21-22', '22-23', '23-24', '24-25', '25-26']

Fold 1: train on ['21-22'] -> test on 22-23
Train shape: (360, 54) | Test shape: (360, 54)
Accuracy: 0.5194 | Macro F1: 0.4339
              precision    recall  f1-score   support

           A       0.42      0.52      0.46       105
           D       0.27      0.12      0.17        80
           H       0.64      0.70      0.67       175

    accuracy                           0.52       360
   macro avg       0.44      0.45      0.43       360
weighted avg       0.49      0.52      0.50       360

Confusion Matrix (rows=actual, cols=predicted):
['A', 'D', 'H']
[[ 55  15  35]
 [ 36  10  34]
 [ 41  12 122]]

Fold 2: train on ['21-22', '22-23'] -> test on 23-24
Train shape: (720, 54) | Test shape: (359, 54)
Accuracy: 0.5571 | Macro F1: 0.4505
              precision    recall  f1-score   support

           A       0.53      0.58      0.55       116
           D       0.23      0.08      0.12

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta):
        unique_seasons = sorted(meta["Season"].unique())

        for i in range(1, len(unique_seasons)):
            train_seasons = unique_seasons[:i]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, test_idx, train_seasons, test_season)


# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")
all_df = all_df.reset_index(drop=True)  # important: validator yields positional-style index, keep it clean

print(f"Full dataset shape: {all_df.shape}")
print(f"Seasons found: {sorted(all_df['Season'].unique())}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_results = []
all_importances = []
class_labels = sorted(all_df["FTR"].unique())  # fixed label order across folds for aggregation

for fold_num, (train_idx, test_idx, train_seasons, test_season) in enumerate(validator.split(all_df), start=1):

    train_df = all_df.loc[train_idx]
    test_df = all_df.loc[test_idx]

    print(f"\n{'='*60}")
    print(f"Fold {fold_num}: train on {train_seasons} -> test on {test_season}")
    print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

    X_train = train_df.drop(columns=drop_cols)
    y_train = train_df["FTR"]

    X_test = test_df.drop(columns=drop_cols)
    y_test = test_df["FTR"]

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f} | Macro F1: {f1_macro:.4f}")
    print(classification_report(y_test, y_pred, labels=class_labels))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(class_labels)
    print(confusion_matrix(y_test, y_pred, labels=class_labels))

    fold_results.append({
        "fold": fold_num,
        "test_season": test_season,
        "train_seasons": train_seasons,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "n_train": len(train_df),
        "n_test": len(test_df),
    })

    all_importances.append(pd.Series(rf.feature_importances_, index=X_train.columns))

# --- Aggregate across folds ---
results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print("WALK-FORWARD SUMMARY")
print(f"{'='*60}")
print(results_df[["fold", "test_season", "n_train", "n_test", "accuracy", "f1_macro"]])

print(f"\nMean accuracy across folds: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
print(f"Mean macro F1 across folds: {results_df['f1_macro'].mean():.4f} (+/- {results_df['f1_macro'].std():.4f})")

# --- Aggregate feature importance across folds ---
importance_df = pd.concat(all_importances, axis=1)
importance_df.columns = [f"fold_{r['fold']}_{r['test_season']}" for r in fold_results]
mean_importance = importance_df.mean(axis=1).sort_values(ascending=False)

print("\nTop 15 features by mean importance across all folds:")
print(mean_importance.head(15))

Full dataset shape: (1799, 148)
Seasons found: ['21-22', '22-23', '23-24', '24-25', '25-26']

Fold 1: train on ['21-22'] -> test on 22-23
Train shape: (360, 148) | Test shape: (360, 148)
Accuracy: 0.5278 | Macro F1: 0.4252
              precision    recall  f1-score   support

           A       0.44      0.58      0.50       105
           D       0.27      0.07      0.12        80
           H       0.61      0.70      0.66       175

    accuracy                           0.53       360
   macro avg       0.44      0.45      0.43       360
weighted avg       0.49      0.53      0.49       360

Confusion Matrix (rows=actual, cols=predicted):
['A', 'D', 'H']
[[ 61   6  38]
 [ 35   6  39]
 [ 42  10 123]]

Fold 2: train on ['21-22', '22-23'] -> test on 23-24
Train shape: (720, 148) | Test shape: (359, 148)
Accuracy: 0.5543 | Macro F1: 0.4662
              precision    recall  f1-score   support

           A       0.53      0.56      0.54       116
           D       0.29      0.13     

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta):
        unique_seasons = sorted(meta["Season"].unique())

        for i in range(1, len(unique_seasons)):
            train_seasons = unique_seasons[:i]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, test_idx, train_seasons, test_season)


# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")
all_df = all_df.reset_index(drop=True)  # important: validator yields positional-style index, keep it clean

print(f"Full dataset shape: {all_df.shape}")
print(f"Seasons found: {sorted(all_df['Season'].unique())}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_results = []
all_importances = []
class_labels = sorted(all_df["FTR"].unique())  # fixed label order across folds for aggregation

for fold_num, (train_idx, test_idx, train_seasons, test_season) in enumerate(validator.split(all_df), start=1):

    train_df = all_df.loc[train_idx]
    test_df = all_df.loc[test_idx]

    print(f"\n{'='*60}")
    print(f"Fold {fold_num}: train on {train_seasons} -> test on {test_season}")
    print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

    X_train = train_df.drop(columns=drop_cols)
    y_train = train_df["FTR"]

    X_test = test_df.drop(columns=drop_cols)
    y_test = test_df["FTR"]

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f} | Macro F1: {f1_macro:.4f}")
    print(classification_report(y_test, y_pred, labels=class_labels))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(class_labels)
    print(confusion_matrix(y_test, y_pred, labels=class_labels))

    fold_results.append({
        "fold": fold_num,
        "test_season": test_season,
        "train_seasons": train_seasons,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "n_train": len(train_df),
        "n_test": len(test_df),
    })

    all_importances.append(pd.Series(rf.feature_importances_, index=X_train.columns))

# --- Aggregate across folds ---
results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print("WALK-FORWARD SUMMARY")
print(f"{'='*60}")
print(results_df[["fold", "test_season", "n_train", "n_test", "accuracy", "f1_macro"]])

print(f"\nMean accuracy across folds: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
print(f"Mean macro F1 across folds: {results_df['f1_macro'].mean():.4f} (+/- {results_df['f1_macro'].std():.4f})")

# --- Aggregate feature importance across folds ---
importance_df = pd.concat(all_importances, axis=1)
importance_df.columns = [f"fold_{r['fold']}_{r['test_season']}" for r in fold_results]
mean_importance = importance_df.mean(axis=1).sort_values(ascending=False)

print("\nTop 15 features by mean importance across all folds:")
print(mean_importance.head(15))

Full dataset shape: (5393, 49)
Seasons found: ['11-12', '12-13', '13-14', '14-15', '15-16', '16-17', '17-18', '18-19', '19-20', '20-21', '21-22', '22-23', '23-24', '24-25', '25-26']

Fold 1: train on ['11-12'] -> test on 12-13
Train shape: (359, 49) | Test shape: (359, 49)
Accuracy: 0.4763 | Macro F1: 0.4391
              precision    recall  f1-score   support

           A       0.41      0.41      0.41       102
           D       0.33      0.26      0.29       101
           H       0.58      0.66      0.62       156

    accuracy                           0.48       359
   macro avg       0.44      0.44      0.44       359
weighted avg       0.46      0.48      0.47       359

Confusion Matrix (rows=actual, cols=predicted):
['A', 'D', 'H']
[[ 42  26  34]
 [ 35  26  40]
 [ 26  27 103]]

Fold 2: train on ['11-12', '12-13'] -> test on 13-14
Train shape: (718, 49) | Test shape: (360, 49)
Accuracy: 0.5139 | Macro F1: 0.4973
              precision    recall  f1-score   support

       

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta, window=4):
        unique_seasons = sorted(meta["Season"].unique())

        for i in range(1, len(unique_seasons)):
            train_seasons = unique_seasons[max(0, i - window):i]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, test_idx, train_seasons, test_season)


# --- Load the full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")
all_df = all_df.reset_index(drop=True)  # important: validator yields positional-style index, keep it clean

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]
class_labels = sorted(all_df["FTR"].unique())  # fixed label order across folds for aggregation


def run_walk_forward(all_df, drop_cols, class_labels, window):
    """Runs a walk-forward loop for one window size and returns per-fold scores."""

    validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))
    fold_results = []

    for fold_num, (train_idx, test_idx, train_seasons, test_season) in enumerate(
        validator.split(all_df, window=window), start=1
    ):
        train_df = all_df.loc[train_idx]
        test_df = all_df.loc[test_idx]

        X_train = train_df.drop(columns=drop_cols).fillna(0)
        y_train = train_df["FTR"]

        X_test = test_df.drop(columns=drop_cols).fillna(0)
        y_test = test_df["FTR"]

        rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)

        fold_results.append({
            "fold": fold_num,
            "test_season": test_season,
            "n_train_seasons": len(train_seasons),
            "n_train": len(train_df),
            "n_test": len(test_df),
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro", labels=class_labels),
        })

    return pd.DataFrame(fold_results)


# --- Sweep training-window sizes 4 through 10 seasons ---
window_summary = []

for window in range(4, 11):
    fold_df = run_walk_forward(all_df, drop_cols, class_labels, window)
    window_summary.append({
        "window": window,
        "mean_accuracy": fold_df["accuracy"].mean(),
        "std_accuracy": fold_df["accuracy"].std(),
        "mean_f1_macro": fold_df["f1_macro"].mean(),
        "std_f1_macro": fold_df["f1_macro"].std(),
        "n_folds": len(fold_df),
    })
    print(
        f"window={window}: mean accuracy={window_summary[-1]['mean_accuracy']:.4f}, "
        f"mean macro F1={window_summary[-1]['mean_f1_macro']:.4f}, folds={len(fold_df)}"
    )

window_summary_df = pd.DataFrame(window_summary)
print("\nWindow sweep summary:")
print(window_summary_df)


# --- Plot: performance vs. training-window size ---
fig, ax = plt.subplots(figsize=(8, 5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

ax.plot(
    window_summary_df["window"], window_summary_df["mean_accuracy"],
    marker="o", markersize=8, linewidth=2, color="#2a78d6", label="Mean accuracy"
)
ax.plot(
    window_summary_df["window"], window_summary_df["mean_f1_macro"],
    marker="o", markersize=8, linewidth=2, color="#008300", label="Mean macro F1"
)

ax.set_xlabel("Training window (seasons)", color="#0b0b0b")
ax.set_ylabel("Score", color="#0b0b0b")
ax.set_title("Random Forest walk-forward performance vs. training window size", color="#0b0b0b")
ax.set_xticks(window_summary_df["window"])

ax.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_color("#c3c2b7")

ax.tick_params(colors="#52514e")
ax.legend(frameon=False, labelcolor="#0b0b0b")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Simple single run: train on the last 4 seasons, test on 25-26 ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")

train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)]
test_df = all_df[all_df["Season"] == test_season]

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols).fillna(0)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols).fillna(0)
y_test = test_df["FTR"]

print(f"Train: {train_seasons} -> {X_train.shape[0]} matches")
print(f"Test: {test_season} -> {X_test.shape[0]} matches")

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(rf.classes_)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# --- Same split as the simple run above (train on last 4 seasons, test on 25-26),
#     but using predict_proba() to get real probabilities instead of a hard label,
#     converted to odds (1/p) and compared against the bookmakers' average odds. ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")

train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)]
test_df = all_df[all_df["Season"] == test_season].copy()

# Exclude the usual identifiers/leakage columns AND the bookmaker odds columns
# themselves -- those are what we're comparing against, not features to train on.
drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies"
]

X_train = train_df.drop(columns=drop_cols).fillna(0)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols).fillna(0)

print(f"Train: {train_seasons} -> {X_train.shape[0]} matches, {X_train.shape[1]} features")
print(f"Test: {test_season} -> {X_test.shape[0]} matches")

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

# --- Convert predicted probabilities into "model odds" ---
proba = rf.predict_proba(X_test)
model_odds = 1 / np.clip(proba, 1e-6, None)  # clip guards against a divide-by-zero on a 0% class

class_to_odds_col = {"H": "ModelHomeOdds", "D": "ModelDrawOdds", "A": "ModelAwayOdds"}
odds_cols_ordered = [class_to_odds_col[c] for c in rf.classes_]
model_odds_df = pd.DataFrame(model_odds, columns=odds_cols_ordered, index=test_df.index)

comparison = pd.concat([
    test_df[["Date", "HomeTeam", "AwayTeam", "FTR", "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds"]],
    model_odds_df[["ModelHomeOdds", "ModelDrawOdds", "ModelAwayOdds"]]
], axis=1)

# The market's odds carry a built-in margin (overround), but the model's
# probabilities sum to exactly 1 -- de-vig the market odds too so the
# comparison isn't just "model odds are bigger because they have no margin".
overround = 1 / comparison["AvgHomeOdds"] + 1 / comparison["AvgDrawOdds"] + 1 / comparison["AvgAwayOdds"]
comparison["FairMarketHomeOdds"] = 1 / (1 / comparison["AvgHomeOdds"] / overround)
comparison["FairMarketDrawOdds"] = 1 / (1 / comparison["AvgDrawOdds"] / overround)
comparison["FairMarketAwayOdds"] = 1 / (1 / comparison["AvgAwayOdds"] / overround)

pd.set_option("display.width", 160)
print("\nSample comparison (first 10 matches):")
print(comparison.head(10).round(2))

print("\n=== Model odds vs. market odds ===")
for outcome, model_col, market_col, fair_col in [
    ("Home", "ModelHomeOdds", "AvgHomeOdds", "FairMarketHomeOdds"),
    ("Draw", "ModelDrawOdds", "AvgDrawOdds", "FairMarketDrawOdds"),
    ("Away", "ModelAwayOdds", "AvgAwayOdds", "FairMarketAwayOdds"),
]:
    mean_model = comparison[model_col].mean()
    mean_market = comparison[market_col].mean()
    mean_fair = comparison[fair_col].mean()
    mae_vs_market = (comparison[model_col] - comparison[market_col]).abs().mean()
    mae_vs_fair = (comparison[model_col] - comparison[fair_col]).abs().mean()
    corr = comparison[model_col].corr(comparison[market_col])
    print(
        f"{outcome:5s}: mean model={mean_model:5.2f}  mean market={mean_market:5.2f}  "
        f"mean fair-market={mean_fair:5.2f}  MAE(vs market)={mae_vs_market:.2f}  "
        f"MAE(vs fair)={mae_vs_fair:.2f}  corr={corr:.3f}"
    )
